# 10 Web map layers

Packages steps 03/04/07's already-cited outputs into small, web-friendly files for docs/. No claims are tagged to this step directly (per steps/10's own Evidence section) -- which underlying claims back each layer is documented in those steps; this notebook only simplifies, reprojects, and exports. City-of-Revelstoke-sourced geometry never appears in any output here (checked explicitly below): only ParcelMap BC and NRCan HRDEM are used for anything published to docs/.

In [ ]:
processed_dir = "data/processed"
raw_dir = "data/raw"
docs_data_dir = "docs/data"
project_crs = "EPSG:26911"
web_crs = "EPSG:4326"

# Geometry simplification tolerance (judgment call): applied in the
# project CRS (metres) before reprojecting to EPSG:4326 for the web, so
# the tolerance means the same real-world distance for every layer.
simplify_tolerance_m = 5

max_file_size_mb = 25

# Elevation-band raster is coarsened before vectorizing (native 1 m HRDEM
# pixels would make an enormous polygon count for a web layer); factor
# is in pixels per side, e.g. 10 means one output pixel per 10x10 block.
elevation_band_coarsen_factor = 10

# Hillshade PNG: HRDEM's native 1 m resolution is far more than a web
# image needs; every Nth pixel is kept in each dimension.
hillshade_downsample_factor = 8

## AOI layer

Step 03's AOI polygon (C063), simplified in the project CRS then reprojected to EPSG:4326 for the web.

In [ ]:
import os

import geopandas as gpd

os.makedirs(docs_data_dir, exist_ok=True)

aoi = gpd.read_file(f"{processed_dir}/03_aoi.gpkg")
# Explicit allowlist (not just the Checks cell's blocklist below) -- defense in depth in case
# 03_aoi.gpkg ever gains a column the blocklist didn't anticipate.
aoi_simplified = aoi[["osm_way_id", "name", "geometry"]].copy()
aoi_simplified["geometry"] = aoi_simplified.geometry.simplify(simplify_tolerance_m)
aoi_web = aoi_simplified.to_crs(web_crs)
aoi_path = f"{docs_data_dir}/10_aoi.geojson"
aoi_web.to_file(aoi_path, driver="GeoJSON")
print(f"{aoi_path}: {os.path.getsize(aoi_path)} bytes")
aoi_web.total_bounds

## Lifts layer

OSM lift geometry (already EPSG:4326, no reprojection needed) from S050's saved Overpass result, joined to step 03's lift crosswalk (data/processed/03_lift_crosswalk.csv) for DEM-derived vertical rise and the proposed master-plan map_ref.

In [ ]:
import json

import pandas as pd
from shapely.geometry import LineString, shape

with open(f"{raw_dir}/S050_overpass_lifts_pistes.json", encoding="utf-8") as f:
    overpass = json.load(f)

lift_crosswalk = pd.read_csv(f"{processed_dir}/03_lift_crosswalk.csv")
crosswalk_by_id = lift_crosswalk.set_index("osm_id")

lift_rows = []
for el in overpass["elements"]:
    if el.get("type") != "way" or "aerialway" not in el.get("tags", {}):
        continue
    if el["id"] not in crosswalk_by_id.index:
        continue
    coords = [(pt["lon"], pt["lat"]) for pt in el["geometry"]]
    row = crosswalk_by_id.loc[el["id"]]
    lift_rows.append({
        "osm_id": el["id"], "name": el["tags"].get("name"),
        "aerialway_type": row["aerialway_type"], "proposed_map_ref": row["proposed_map_ref"],
        "dem_vert_rise_m": row["dem_vert_rise_m"], "vert_rise_delta_m": row["vert_rise_delta_m"],
        "geometry": LineString(coords),
    })
lifts = gpd.GeoDataFrame(lift_rows, crs=web_crs)
print(f"{len(lifts)} lifts (of {len(lift_crosswalk)} in the crosswalk)")

lifts_projected = lifts.to_crs(project_crs)
lifts_projected["geometry"] = lifts_projected.geometry.simplify(simplify_tolerance_m)
lifts_web = lifts_projected.to_crs(web_crs)
lifts_path = f"{docs_data_dir}/10_lifts.geojson"
lifts_web.to_file(lifts_path, driver="GeoJSON")
print(f"{lifts_path}: {os.path.getsize(lifts_path)} bytes")

## Runs layer

Step 03 already computed per-run difficulty and DEM slope stats (data/processed/03_runs.gpkg); this just simplifies and reprojects.

In [ ]:
runs = gpd.read_file(f"{processed_dir}/03_runs.gpkg")
# Explicit allowlist, same reasoning as the AOI layer above.
runs_keep_cols = ["osm_id", "name", "osm_difficulty", "length_m", "vert_drop_m",
                   "mean_slope_deg", "max_slope_deg", "geometry"]
runs_simplified = runs[runs_keep_cols].copy()
runs_simplified["geometry"] = runs_simplified.geometry.simplify(simplify_tolerance_m)
runs_web = runs_simplified.to_crs(web_crs)
runs_path = f"{docs_data_dir}/10_runs.geojson"
runs_web.to_file(runs_path, driver="GeoJSON")
print(f"{runs_path}: {os.path.getsize(runs_path)} bytes, {len(runs_web)} runs")

## Elevation bands layer: rebuild the band raster

Step 04 only wrote an area-by-band table (data/processed/04_elevation_bands.csv), not polygons, so the band raster is rebuilt here from the same DEM clip and the same band edges, then vectorized below. Coarsened first (elevation_band_coarsen_factor) since native 1 m pixels would produce an unusably large polygon count for a simplified web layer.

In [ ]:
import sys

import numpy as np

sys.path.insert(0, "src")
from resort.dem import clip_dem

bands = pd.read_csv(f"{processed_dir}/04_elevation_bands.csv")
band_edges = bands["bottom_m"].tolist() + [bands["top_m"].iloc[-1]]
band_labels = bands["band"].tolist()

dem = clip_dem(tuple(aoi.total_bounds), project_crs, buffer_m=0)
dem_coarse = dem.coarsen(
    x=elevation_band_coarsen_factor, y=elevation_band_coarsen_factor, boundary="trim"
).mean()
elev_arr = dem_coarse.values[0]
# recalc=True is required: xarray's coarsen().mean() is CRS-unaware and never updates
# the cached GeoTransform on the array's spatial_ref coordinate, so the default
# .rio.transform() silently hands back the PRE-coarsen (1 m) transform even though the
# array is now coarser -- caught by notebook-reviewer, verified directly against the DEM.
coarse_transform = dem_coarse.rio.transform(recalc=True)
print(f"coarsened DEM shape: {elev_arr.shape} (from {dem.values[0].shape})")

## Vectorize, dissolve, simplify, export


In [ ]:
from rasterio.features import shapes as rio_shapes

band_idx = np.digitize(elev_arr, band_edges) - 1
band_idx = np.clip(band_idx, 0, len(band_labels) - 1).astype("int32")
band_idx[np.isnan(elev_arr)] = -1

band_polygons = []
for geom, value in rio_shapes(band_idx, mask=(band_idx != -1), transform=coarse_transform):
    band_polygons.append({"band": band_labels[int(value)], "geometry": shape(geom)})

band_gdf = gpd.GeoDataFrame(band_polygons, crs=project_crs)
elevation_bands = band_gdf.dissolve(by="band", as_index=False)
# clip_dem's bounds are the AOI's rectangular bounding box, not its actual polygon shape, so
# without this the band polygons spill out well beyond the resort boundary into surrounding
# terrain -- clip to the real AOI polygon to match step 04's own area accounting.
aoi_union = aoi.geometry.union_all()
elevation_bands["geometry"] = elevation_bands.geometry.intersection(aoi_union)
elevation_bands = elevation_bands[~elevation_bands.geometry.is_empty]
elevation_bands["geometry"] = elevation_bands.geometry.simplify(simplify_tolerance_m)
elevation_bands_web = elevation_bands.to_crs(web_crs)
bands_path = f"{docs_data_dir}/10_elevation_bands.geojson"
elevation_bands_web.to_file(bands_path, driver="GeoJSON")
print(f"{bands_path}: {os.path.getsize(bands_path)} bytes, {len(elevation_bands_web)} band polygons")

## Funnel parcels layer

step 07 wrote per-parcel results only for the parcels that survived its full filter chain (data/processed/07_network_distances.csv, 150 PIDs) -- not a stage-reached attribute for every candidate parcel, since that table was never exported. This layer therefore publishes only "passed the full funnel" parcels, not a richer pass/fail-per-stage breakdown; noted in steps/07's Open issues as a gap for a future revisit, not silently narrowed here.

Geometry is ParcelMap BC (S030/C065), not the City's own parcel fabric, per this step's own method and the licence rule.

In [ ]:
from resort.arcgis import fetch_parcelmap_bc

network_distances = pd.read_csv(f"{processed_dir}/07_network_distances.csv", dtype={"PID": str})
survivor_pids = set(network_distances["PID"].dropna())
print(f"{len(survivor_pids)} funnel-surviving parcels with a real PID "
      f"(of {len(network_distances)} rows -- some ParcelMap BC parcels have no PID)")

all_city_parcels = fetch_parcelmap_bc("MUNICIPALITY='Revelstoke, City of'", out_crs=project_crs)
funnel_parcels = all_city_parcels[all_city_parcels["PID"].isin(survivor_pids)].merge(
    network_distances, on="PID", how="left"
)
funnel_parcels["passed_full_funnel"] = True
print(f"matched {len(funnel_parcels)} of {len(survivor_pids)} survivor PIDs to live ParcelMap BC geometry")

## Simplify, reproject, export funnel parcels


In [ ]:
keep_cols = [
    "PID", "PARCEL_CLASS", "FEATURE_AREA_SQM", "passed_full_funnel",
    "network_dist_to_gondola_base_m", "network_dist_to_downtown_m", "geometry",
]
funnel_parcels_out = funnel_parcels[keep_cols].copy()
funnel_parcels_out["geometry"] = funnel_parcels_out.geometry.simplify(simplify_tolerance_m)
funnel_parcels_web = funnel_parcels_out.to_crs(web_crs)
funnel_path = f"{docs_data_dir}/10_funnel_parcels.geojson"
funnel_parcels_web.to_file(funnel_path, driver="GeoJSON")
print(f"{funnel_path}: {os.path.getsize(funnel_path)} bytes, {len(funnel_parcels_web)} parcels")

## Hillshade image

NRCan's HRDEM STAC item publishes a pre-rendered hillshade-dtm asset alongside the dtm asset itself; clipping that (a windowed read, same clip_dem helper, just a different URL) needs no new hillshade computation.

In [ ]:
HILLSHADE_URL = (
    "https://canelevation-dem.s3.ca-central-1.amazonaws.com/"
    "hrdem-mosaic-1m/2_4-mosaic-1m-dtm_hillshade.tif"
)
hillshade = clip_dem(tuple(aoi.total_bounds), project_crs, buffer_m=200, url=HILLSHADE_URL)
hillshade_arr = hillshade.values[0]
hillshade_small = hillshade_arr[::hillshade_downsample_factor, ::hillshade_downsample_factor]
print(f"hillshade array: {hillshade_arr.shape} -> downsampled to {hillshade_small.shape}")

## Write hillshade PNG


In [ ]:
import matplotlib.pyplot as plt

hillshade_path = f"{docs_data_dir}/10_hillshade.png"
plt.imsave(hillshade_path, np.nan_to_num(hillshade_small, nan=0), cmap="gray")
print(f"{hillshade_path}: {os.path.getsize(hillshade_path)} bytes")

## Map: all exported web layers together

A visual check that every layer lines up spatially before publishing, rendered over the clipped hillshade.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
extent = hillshade.rio.bounds()
ax.imshow(
    np.nan_to_num(hillshade_arr, nan=np.nanmin(hillshade_arr)), cmap="gray",
    extent=(extent[0], extent[2], extent[1], extent[3]), origin="upper",
)
aoi.boundary.plot(ax=ax, color="black", linewidth=1.2, label="AOI")
elevation_bands.plot(ax=ax, column="band", alpha=0.35, legend=True)
runs.plot(ax=ax, color="blue", linewidth=0.4)
lifts_projected.plot(ax=ax, color="red", linewidth=1.5)
funnel_parcels.plot(ax=ax, color="lime", edgecolor="black", linewidth=0.3)
ax.set_title("Step 10 web layers: AOI, elevation bands, runs, lifts, funnel parcels")
ax.set_xlabel(f"Easting ({project_crs})")
ax.set_ylabel("Northing")
plt.tight_layout()
plt.show()

## Checks

Every file's size reported and asserted under max_file_size_mb; no City-of-Revelstoke-specific field name appears in any exported GeoJSON (a real grep-style check against field names unique to the City's own FeatureServers, not just a claim it was avoided); every GeoJSON's coordinates fall in plausible longitude/latitude ranges for the Revelstoke area.

In [ ]:
# Independent-number check: elevation-band areas should match step 04's own area accounting
# for the identical AOI and band edges (data/processed/04_elevation_bands.csv only has area
# totals, not polygons, which is exactly why this notebook rebuilds them -- so this is a real
# cross-check between two independently-computed area figures, not a tautology).
elevation_bands_project = elevation_bands.to_crs(project_crs)
band_area_ha = (elevation_bands_project.geometry.area / 10_000).sum()
step04_bands = pd.read_csv(f"{processed_dir}/04_elevation_bands.csv")
aoi_area_ha = aoi.geometry.area.iloc[0] / 10_000
print(f"step 10 rebuilt band polygons total: {band_area_ha:.1f} ha; step 03/04's own AOI area: {aoi_area_ha:.1f} ha")
assert abs(band_area_ha - aoi_area_ha) / aoi_area_ha < 0.05, (
    "rebuilt elevation-band area doesn't match step 04's AOI area within 5% -- "
    "likely the same coarsen/transform/clip issue notebook-reviewer caught once already"
)

geojson_paths = [aoi_path, lifts_path, runs_path, bands_path, funnel_path]
image_paths = [hillshade_path]

for path in geojson_paths + image_paths:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    assert size_mb < max_file_size_mb, f"{path} is {size_mb:.2f} MB, over the {max_file_size_mb} MB limit"
print("all files under the size limit")

# City-only field names: none of these should appear as a property key in any exported GeoJSON.
city_only_fields = {
    "zoningName", "ocpHazardType", "assetCORID", "assetType", "assetMaterial",
    "assetDiameter", "roadName", "roadClass", "roadStatus", "roadWidth", "GlobalID",
}
for path in geojson_paths:
    with open(path, encoding="utf-8") as f:
        gj = json.load(f)
    all_keys = set()
    for feat in gj["features"]:
        all_keys.update(feat["properties"].keys())
    leaked = all_keys & city_only_fields
    assert not leaked, f"{path} contains City-only field(s): {leaked}"
print("no City-of-Revelstoke field names found in any exported layer")

for path in geojson_paths:
    with open(path, encoding="utf-8") as f:
        gj = json.load(f)
    for feat in gj["features"]:
        geom = shape(feat["geometry"])
        minx, miny, maxx, maxy = geom.bounds
        assert -119.5 < minx < -117.0 and -119.5 < maxx < -117.0, (path, minx, maxx)
        assert 50.0 < miny < 51.5 and 50.0 < maxy < 51.5, (path, miny, maxy)
print("all coordinates fall in plausible Revelstoke-area longitude/latitude ranges")
print("checks passed")

## Versions

In [ ]:
import importlib.metadata

for pkg in ["pandas", "geopandas", "shapely", "pyproj", "rasterio", "rioxarray", "numpy", "matplotlib"]:
    print(pkg, importlib.metadata.version(pkg))